# A diffusion image generator

_Capstones_

**Stitch four papers into one small conditional diffusion model that denoises pure noise into toy images — and steer what it draws with classifier-free guidance.**

---

This notebook is a practice scaffold for the **A diffusion image generator** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Fix the forward noising schedule and the toy dataA DDPM has a *fixed* forward process that gradually turns data into noise — no learning involved. We define a small linear `beta` variance schedule over `T = 50` steps, then derive `alpha_t = 1 - beta_t` and the cumulative product `alpha-bar_t`, which lets us jump to any noise level in one shot. Our "images" are a ring of 8 Gaussian clusters; clusters on the right half are class 0, on the left half class 1.

In [ ]:
import torch
import torch.nn as nn
import math
torch.manual_seed(0)

# --- The forward noising schedule (paper-ddpm). A fixed recipe, no learning. ---
T = 50
betas = torch.linspace(1e-4, 0.02, T)              # small linear variance schedule
alphas = 1 - betas
abar = torch.cumprod(alphas, 0)                    # alpha-bar_t = prod_{s<=t} alpha_s

# --- The toy data: a ring of 8 clusters; right half = class 0, left half = class 1. ---
def sample_data(n):
    k = torch.randint(0, 8, (n,))                  # which of the 8 clusters
    ang = k.float() / 8 * 2 * math.pi
    centers = torch.stack([torch.cos(ang), torch.sin(ang)], 1) * 2.0
    x = centers + torch.randn(n, 2) * 0.15
    c = (centers[:, 0] < 0).long()                 # class 1 if cluster is on the left (x<0), else class 0
    return x, c

### Step 2 — Define the U-Net-shaped noise predictorThe network learns to predict the noise `eps_theta(x_t, t, c)` that was added at step `t`. It is a small MLP with a U-Net shape — an encoder (down), a bottleneck (mid), and a decoder (up) that receives a skip connection from the encoder. A class embedding table has three rows: class 0, class 1, and a **NULL** token used by classifier-free guidance to represent "no condition".

In [ ]:
# --- The noise predictor eps_theta(x_t, t, c): a U-NET-shaped MLP (paper-unet) ---
#     with a class embedding that has a NULL row (paper-cfg). down -> bottleneck -> up + skip.
NULL = 2                                            # ids 0,1 = classes; id 2 = null token (no condition)

class UNetDenoiser(nn.Module):
    def __init__(self, h=128):
        super().__init__()
        self.emb = nn.Embedding(3, 16)              # 3 rows: class 0, class 1, NULL
        self.down = nn.Sequential(nn.Linear(2 + 1 + 16, h), nn.SiLU())   # encoder (down)
        self.mid = nn.Sequential(nn.Linear(h, h), nn.SiLU())            # bottleneck
        self.up = nn.Sequential(nn.Linear(h + h, h), nn.SiLU())         # decoder (up), takes the skip
        self.out = nn.Linear(h, 2)

    def forward(self, x, t, c):
        te = (t.float() / T).unsqueeze(1)           # timestep, scaled to [0,1]
        cond = self.emb(c)                          # class embedding (or NULL)
        d = self.down(torch.cat([x, te, cond], 1))
        m = self.mid(d)
        u = self.up(torch.cat([m, d], 1))           # skip connection: hand 'd' across to the decoder
        return self.out(u)

### Step 3 — Train: predict the noise, dropping the label sometimesTraining follows the DDPM noise-MSE objective (Eq. 14): pick random examples and random timesteps, add the corresponding noise in one shot via the reparameterization (Eq. 4), and have the network predict that noise. The classifier-free-guidance twist (Alg. 1) is that with probability `p_uncond` we replace the true label with the NULL token, so the *same* network learns both a conditional and an unconditional denoiser.

In [ ]:
# --- Training (paper-ddpm Eq. 14 + paper-cfg Alg. 1): noise-MSE, drop the label sometimes. ---
p_uncond = 0.2

def train(steps=4000):
    torch.manual_seed(0)
    net = UNetDenoiser()
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    for step in range(steps):
        x0, c = sample_data(512)
        drop = torch.rand(512) < p_uncond                        # paper-cfg Alg. 1: with prob p_uncond...
        c_in = torch.where(drop, torch.full_like(c, NULL), c)    # ...replace the label by the NULL token
        tb = torch.randint(0, T, (512,))                         # random timesteps (paper-ddpm Alg. 1)
        eps = torch.randn_like(x0)                               # the noise to predict
        ab = abar[tb].unsqueeze(1)
        xt = ab.sqrt() * x0 + (1 - ab).sqrt() * eps              # Eq. 4: forward-noise in one shot (reparam.)
        pred = net(xt, tb, c_in)
        loss = ((eps - pred) ** 2).mean()                        # Eq. 14: L_simple (predict the noise)
        opt.zero_grad()
        loss.backward()
        opt.step()
        if step % 1000 == 0:
            print(f"  step {step:4d}  loss {loss.item():.4f}")
    return net

### Step 4 — Guided sampling, then train and probe the modelSampling reverses the process (Alg. 2): start from pure noise `x_T` and denoise step by step. At each step we run the network **twice** — once conditioned on the class, once on the NULL token — and mix them with the guidance scale `w` (Eq. 6) to push samples toward the chosen class. We then train the model and run three probes: watch the mean radius climb toward the ring as noise is removed, sweep `w` to see guidance pull class-1 samples onto their (left-half) clusters, and an ablation contrasting `w=0` with `w=3`.

In [ ]:
# --- Guided sampling (paper-ddpm Alg. 2 + paper-cfg Eq. 6): two predictions/step, mix, step. ---
@torch.no_grad()
def sample(net, n, c_val, w, snapshot_at=()):
    x = torch.randn(n, 2)                                       # x_T ~ N(0, I): pure noise
    c = torch.full((n,), c_val, dtype=torch.long)
    cn = torch.full((n,), NULL, dtype=torch.long)              # null token for the label-blind pass
    snaps = {}
    for t in reversed(range(T)):                               # t = T-1, ..., 0
        tb = torch.full((n,), t, dtype=torch.long)
        ec = net(x, tb, c)                                     # conditional   eps_theta(x, t, c)
        eu = net(x, tb, cn)                                    # unconditional eps_theta(x, t)
        e = (1 + w) * ec - w * eu                              # paper-cfg Eq. 6: guided noise estimate
        a = alphas[t]
        ab = abar[t]
        mean = (1 / a.sqrt()) * (x - ((1 - a) / (1 - ab).sqrt()) * e)   # paper-ddpm Alg. 2 denoising step
        if t > 0:
            x = mean + betas[t].sqrt() * torch.randn_like(x)
        else:
            x = mean                                           # no noise added at the last step
        if t in snapshot_at:
            snaps[t] = x.clone()
    return x, snaps


# --- Train, then PRINT samples emerging from noise (class 1, mild guidance w=1). ---
print("TRAIN the conditional DDPM:")
net = train()

print("\nPRINT samples emerging from noise (class 1, w=1) -- mean radius should climb toward 2.0:")
_, snaps = sample(net, 600, c_val=1, w=1.0, snapshot_at=(49, 40, 30, 20, 10, 0))
for t in (49, 40, 30, 20, 10, 0):
    r = snaps[t].norm(dim=1).mean().item()
    print(f"  t={t:2d}  mean radius {r:.3f}")
# Typical: radius climbs from ~1.4 (a near-pure-noise blob) toward ~2.0 (seated on the ring).
# (Our small run -- not a paper's reported number.)


# --- PRINT guidance sharpening: sweep w for class 1 (its clusters are all on the LEFT, x<0). ---
#     Higher w => samples land more firmly on the correct (left) half and tighter on the ring.
ang = torch.arange(8).float() / 8 * 2 * math.pi
modes = torch.stack([torch.cos(ang), torch.sin(ang)], 1) * 2.0          # the 8 cluster centers
left = modes[modes[:, 0] < 0]                                          # the 4 class-1 (left) centers
print("\nGUIDANCE SWEEP, class 1 (clusters on the left, x<0):")
for w in (0.0, 1.0, 3.0):
    s, _ = sample(net, 3000, c_val=1, w=w)
    frac_left = (s[:, 0] < 0).float().mean().item()                    # fraction on the correct half
    d_left = torch.cdist(s, left).min(1).values                        # distance to nearest class-1 cluster
    on_mode = (d_left < 0.4).float().mean().item()                     # fraction tight on a class-1 cluster
    print(f"  w={w:.0f}:  frac on class-1 half {frac_left:.3f}   frac on a class-1 cluster {on_mode:.3f}")
# Typical (our small run, not a paper's number): both fractions rise as w grows --
# guidance pulls samples onto the conditioned class and sharpens them onto its clusters.

# --- ABLATION: w=0 (plain conditional) vs w=3 (strong guidance) -- isolates the guidance term. ---
print("\nABLATION (w=0 = plain conditional model; w=3 = strong guidance):")
for w in (0.0, 3.0):
    s, _ = sample(net, 3000, c_val=1, w=w)
    frac_left = (s[:, 0] < 0).float().mean().item()
    print(f"  w={w:.0f}:  frac on class-1 half {frac_left:.3f}")
# w=0 leaks some samples onto the wrong (right) half; w=3 concentrates them on the class-1 half.

## Visualize the data & results

_Does the assembled conditional DDPM denoise pure noise into the ring of clusters, and does raising the guidance scale w pull class-1 samples onto their (left-half) clusters?_

### Step 1 — Rebuild the schedule, data, and denoiserThis visualization cell is self-contained, so it re-imports torch, re-seeds, and rebuilds the same forward schedule, toy data, and U-Net denoiser as the reference implementation above. Reconstructing them here means the plotting run does not depend on any variables left over from earlier cells.

In [ ]:
import torch
import torch.nn as nn
import math
torch.manual_seed(0)

# --- forward schedule (paper-ddpm) ---
T = 50
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1 - betas
abar = torch.cumprod(alphas, 0)

# --- toy data: ring of 8 clusters, right half = class 0, left half = class 1 ---
def sample_data(n):
    k = torch.randint(0, 8, (n,))
    ang = k.float() / 8 * 2 * math.pi
    centers = torch.stack([torch.cos(ang), torch.sin(ang)], 1) * 2.0
    x = centers + torch.randn(n, 2) * 0.15
    return x, (centers[:, 0] < 0).long()

# --- U-Net-shaped noise predictor with a NULL embedding row (paper-unet + paper-cfg) ---
NULL = 2

class UNetDenoiser(nn.Module):
    def __init__(self, h=128):
        super().__init__()
        self.emb = nn.Embedding(3, 16)
        self.down = nn.Sequential(nn.Linear(2 + 1 + 16, h), nn.SiLU())
        self.mid = nn.Sequential(nn.Linear(h, h), nn.SiLU())
        self.up = nn.Sequential(nn.Linear(h + h, h), nn.SiLU())
        self.out = nn.Linear(h, 2)

    def forward(self, x, t, c):
        te = (t.float() / T).unsqueeze(1)
        d = self.down(torch.cat([x, te, self.emb(c)], 1))
        m = self.mid(d)
        u = self.up(torch.cat([m, d], 1))           # skip connection
        return self.out(u)

### Step 2 — Train the model for plottingWe train a fresh denoiser with the same noise-MSE objective (Eq. 14) and the same label-dropping for classifier-free guidance (Alg. 1) as before. This is the model whose samples we will visualize.

In [ ]:
# --- train: noise-MSE (Eq. 14) + label drop (paper-cfg Alg. 1) ---
net = UNetDenoiser()
opt = torch.optim.Adam(net.parameters(), lr=1e-3)
for _ in range(4000):
    x0, c = sample_data(512)
    drop = torch.rand(512) < 0.2
    c_in = torch.where(drop, torch.full_like(c, NULL), c)
    tb = torch.randint(0, T, (512,))
    eps = torch.randn_like(x0)
    ab = abar[tb].unsqueeze(1)
    xt = ab.sqrt() * x0 + (1 - ab).sqrt() * eps
    loss = ((eps - net(xt, tb, c_in)) ** 2).mean()
    opt.zero_grad()
    loss.backward()
    opt.step()

### Step 3 — Sample with strong guidance and snapshot the cloudFinally we run guided sampling (Alg. 2 + Eq. 6) for class 1 at a strong guidance scale `w = 3`, capturing snapshots of the point cloud at a few timesteps (`t = 49, 20, 0`). These snapshots are the noise-to-ring progression that the surrounding lesson plots.

In [ ]:
# --- guided sampling (paper-ddpm Alg. 2 + paper-cfg Eq. 6), snapshot the cloud ---
@torch.no_grad()
def sample(n, c_val, w, snap=()):
    x = torch.randn(n, 2)
    c = torch.full((n,), c_val, dtype=torch.long)
    cn = torch.full((n,), NULL, dtype=torch.long)
    snaps = {}
    for t in reversed(range(T)):
        tb = torch.full((n,), t, dtype=torch.long)
        ec = net(x, tb, c)
        eu = net(x, tb, cn)
        e = (1 + w) * ec - w * eu                   # Eq. 6
        a = alphas[t]
        ab = abar[t]
        mean = (1 / a.sqrt()) * (x - ((1 - a) / (1 - ab).sqrt()) * e)
        if t > 0:
            x = mean + betas[t].sqrt() * torch.randn_like(x)
        else:
            x = mean
        if t in snap:
            snaps[t] = x[:40].clone()
    return x, snaps

torch.manual_seed(7)
_, snaps = sample(600, c_val=1, w=3.0, snap=(49, 20, 0))   # the points plotted above